In [8]:
"""
Step 3: Build DPO training pairs from:
  - step1_labeling_sheet.csv   (your real labeled papers, filled in)
  - synthetic_negatives.csv    (your reviewed synthetic irrelevant papers)

Output: dpo_pairs.jsonl, one JSON object per line, ready for a DPO trainer
(e.g. Hugging Face TRL's DPOTrainer / DPOConfig).

Each line has: prompt, chosen, rejected

--------------------------------------------------------------------------
HOW REASONS ARE BUILT (read this before running)
--------------------------------------------------------------------------
For each row, the "chosen" reason is built in this priority order:

  1. If the 'reason' column has text in it, that exact text is used as-is.
     Use this when a paper needs a nuanced explanation the structured
     columns can't capture cleanly (e.g. the chicken-thigh / faba-bean
     "which protein is actually being studied" cases).

  2. Otherwise, if the structured columns are filled in
     (protein_source_type, protein_source_specific, isolate_concentrate_flour,
     modification_step, application_property), a sentence is auto-built from
     them in the same "header: value" style you've been writing by hand,
     e.g. "protein source: pea; form: isolate; modification step: hydrolysis;
     application: gelation"

  3. Otherwise (nothing filled in), a generic fallback template is used.

This means: fill in the structured columns as your default, and only use
the free-text 'reason' column for the harder edge cases that need it.
--------------------------------------------------------------------------
"""

import json
import pandas as pd

INSTRUCTION = (
    "Classify this paper as relevant or irrelevant to plant protein functional "
    "properties research. Give a one to two sentence reason naming which of these "
    "criteria are present or missing: plant protein source, functional/physicochemical "
    "property, extraction or modification method.\n\n"
    "Title: {title}\n\nAbstract: {abstract}"
)

# ---- Fallback templates (only used if BOTH reason and structured columns are empty) ----
REASON_RELEVANT_CORRECT = (
    "Relevant. Plant protein source, functional property, and modification/extraction "
    "method are present."
)
REASON_RELEVANT_WRONG_REJECT = (
    "Irrelevant. Insufficient functional property language is present in the title."
)

REASON_IRRELEVANT_CORRECT_TEMPLATE = "Irrelevant. {reason}"
REASON_IRRELEVANT_WRONG_REJECT = (
    "Relevant. Contains plant protein source and modification method keywords."
)

TRAP_TYPE_REASONS = {
    "animal_source": "The protein source is animal-derived, not plant-derived.",
    "insect_source": "The protein source is insect-derived, not plant-derived.",
    "microbial_source": "The protein source is microbial, not plant-derived.",
    "fish_or_seafood_source": "The protein source is fish/seafood-derived, not plant-derived.",
    "poultry_source": "The protein source is poultry-derived, not plant-derived.",
    "wrong_property_focus": (
        "The paper focuses on composition or preservation, not a functional/"
        "physicochemical property."
    ),
    "non_protein_material": (
        "The material studied is a non-protein component (e.g. mucilage, starch, "
        "fiber, gum, pectin, oil), not a protein isolate or concentrate."
    ),
    "fungal_source": "The protein source is fungal-derived, not plant-derived.",
    "algae_source": (
        "The protein source is algae-derived. Decide per your project scope whether "
        "algae counts as plant-adjacent (relevant) or excluded (irrelevant) and edit "
        "this reason accordingly."
    ),
    "plant_protein_as_additive_only": (
        "A plant protein is present but only as an additive; the functional "
        "properties measured belong to a different, non-plant primary protein system."
    ),
}


def _clean(v):
    if v is None:
        return None
    if isinstance(v, float):  # NaN from empty CSV cells
        return None
    v = str(v).strip()
    if not v:
        return None
    if v.lower() in ("none", "n/a", "na", "not applicable", "nil", "-"):
        return None
    return v


def build_wrong_reject_for_relevant(row) -> str:
    """For a paper that IS relevant, build a plausible-sounding WRONG (irrelevant)
    verdict using the SAME structured 'field: value' style as build_reason_from_columns,
    just with the wrong label and one field flipped/misrepresented. Matching format
    prevents the model from learning to distinguish chosen/rejected by writing
    style alone -- only actual content correctness should differ."""
    source = _clean(getattr(row, "source_specific", None)) or _clean(
        getattr(row, "source_type", None)
    )
    mod = _clean(getattr(row, "modification_step", None))
    app = _clean(getattr(row, "application_property", None))
    first_prop = app.split(",")[0].strip() if app else None

    if not mod and app:
        parts = [f"source: {source}"] if source else []
        parts.append("modification step: none identified")
        parts.append(f"application: {first_prop}")
        return "Irrelevant. " + "; ".join(parts)
    elif not app and mod:
        parts = [f"source: {source}"] if source else []
        parts.append(f"modification step: {mod}")
        parts.append("application: no functional property outcome reported")
        return "Irrelevant. " + "; ".join(parts)
    elif not source and (mod or app):
        parts = ["source: not clearly plant-derived"]
        if mod:
            parts.append(f"modification step: {mod}")
        if first_prop:
            parts.append(f"application: {first_prop}")
        return "Irrelevant. " + "; ".join(parts)
    elif source and mod and app:
        # All three genuinely present -- flip via a mislabeled application
        # (the property belongs to a downstream product, not the protein).
        parts = [
            f"source: {source}",
            f"modification step: {mod}",
            f"application: {first_prop} (downstream product property, not the isolated protein)",
        ]
        return "Irrelevant. " + "; ".join(parts)
    else:
        return REASON_RELEVANT_WRONG_REJECT


def build_wrong_reject_for_irrelevant(row, trap_type: str = None) -> str:
    """For a paper that IS irrelevant, build a plausible-sounding WRONG (relevant)
    verdict using the SAME structured 'field: value' style as build_reason_from_columns.
    Where trap_type is known, the flipped field reflects that specific near-miss
    (e.g. a source-based trap mislabels the source field as plant), so the model
    learns each trap's actual blind spot rather than a generic keyword match --
    and cannot use response format/style as a shortcut, since both chosen and
    rejected now share identical structure."""
    source = _clean(getattr(row, "source_specific", None)) or _clean(
        getattr(row, "source_type", None)
    )
    mod = _clean(getattr(row, "modification_step", None))
    app = _clean(getattr(row, "application_property", None))
    first_prop = app.split(",")[0].strip() if app else None

    non_plant_source_traps = {
        "animal_source", "poultry_source", "fish_or_seafood_source",
        "insect_source", "microbial_source", "fungal_source", "algae_source",
    }

    parts = []

    if trap_type in non_plant_source_traps and (mod or first_prop):
        parts.append(f"source: {source} (misread as plant-derived)" if source else "source: plant (misread)")
    elif trap_type == "non_protein_material" and (mod or first_prop):
        parts.append(f"source: {source} (misread as protein isolate)" if source else "source: protein (misread)")
    elif trap_type == "wrong_property_focus" and source:
        parts.append(f"source: {source}")
    elif trap_type == "plant_protein_as_additive_only" and source:
        parts.append(f"source: {source} (misread as primary protein, not an additive)")
    elif trap_type == "no_modification_method" and (source or first_prop):
        if source:
            parts.append(f"source: {source}")
    else:
        if source:
            parts.append(f"source: {source}")

    if mod:
        parts.append(f"modification step: {mod}")
    elif trap_type == "no_modification_method":
        parts.append("modification step: comparative design (misread as satisfying criterion)")

    if first_prop:
        if trap_type == "wrong_property_focus":
            parts.append(f"application: {first_prop} (misread as protein property, actually downstream)")
        else:
            parts.append(f"application: {first_prop}")

    if parts:
        return "Relevant. " + "; ".join(parts)
    else:
        return REASON_IRRELEVANT_WRONG_REJECT


def build_reason_from_columns(row) -> str:
    """Assemble a 'header: value' sentence from structured columns.
    Returns None if nothing usable is present."""
    parts = []

    source = _clean(getattr(row, "source_specific", None)) or _clean(
        getattr(row, "source_type", None)
    )
    if source:
        parts.append(f"source: {source}")

    form = _clean(getattr(row, "isolate_concentrate_flour", None))
    if form:
        parts.append(f"form: {form}")

    mod = _clean(getattr(row, "modification_step", None))
    if mod:
        parts.append(f"modification step: {mod}")

    app = _clean(getattr(row, "application_property", None))
    if app:
        parts.append(f"application: {app}")

    if not parts:
        return None
    return "; ".join(parts)


def build_pair_relevant(title: str, abstract: str, row=None, source_tag="real") -> dict:
    prompt = INSTRUCTION.format(title=title, abstract=abstract)

    custom_reason = _clean(getattr(row, "reason", None)) if row is not None else None
    if custom_reason:
        chosen = f"Relevant. {custom_reason}"
    else:
        auto = build_reason_from_columns(row) if row is not None else None
        chosen = f"Relevant. {auto}" if auto else REASON_RELEVANT_CORRECT

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": build_wrong_reject_for_relevant(row) if row is not None else REASON_RELEVANT_WRONG_REJECT,
        "meta_source": source_tag,
        "meta_trap_type": None,
    }


def build_pair_irrelevant(
    title: str, abstract: str, trap_type: str = None, row=None, source_tag="real"
) -> dict:
    prompt = INSTRUCTION.format(title=title, abstract=abstract)

    custom_reason = _clean(getattr(row, "reason", None)) if row is not None else None
    if custom_reason:
        chosen = f"Irrelevant. {custom_reason}"
    else:
        auto = build_reason_from_columns(row) if row is not None else None
        if auto:
            chosen = f"Irrelevant. {auto}"
        else:
            reason = TRAP_TYPE_REASONS.get(trap_type, "It does not meet the relevance criteria.")
            chosen = REASON_IRRELEVANT_CORRECT_TEMPLATE.format(reason=reason)

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": build_wrong_reject_for_irrelevant(row, trap_type) if row is not None else REASON_IRRELEVANT_WRONG_REJECT,
        "meta_source": source_tag,
        "meta_trap_type": trap_type,
    }


def _read_table(path):
    """Read either an Excel file (.xlsx/.xls) or a CSV, trying UTF-8 first
    and falling back to Windows-1252 for CSVs saved/edited in Excel on Windows
    that contain characters like en-dashes or curly quotes."""
    if str(path).lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def main():
    pairs = []

    real = _read_table("step1_labeling_sheet.xlsx")
    real["label"] = real["label"].astype(str).str.strip().str.lower()
    valid = real["label"].isin(["relevant", "irrelevant"])
    if (~valid).any():
        print(f"WARNING: {(~valid).sum()} rows have no valid label, skipping them.")
    real = real[valid]

    trap_col_present = "trap_type" in real.columns

    for row in real.itertuples():
        if row.label == "relevant":
            pairs.append(build_pair_relevant(row.title, row.abstract, row, source_tag="real"))
        else:
            trap = getattr(row, "trap_type", None) if trap_col_present else None
            trap = _clean(trap)
            pairs.append(build_pair_irrelevant(row.title, row.abstract, trap, row, source_tag="real"))

    print(f"Built {len(pairs)} pairs from real labeled papers.")

    try:
        synth = _read_table("synthetic_negatives.csv")
        for row in synth.itertuples():
            trap = _clean(getattr(row, "trap_type", None))
            pairs.append(build_pair_irrelevant(row.title, row.abstract, trap, row, source_tag="synthetic"))
        print(f"Added {len(synth)} pairs from synthetic negatives.")
    except FileNotFoundError:
        print("No synthetic_negatives.csv found yet, skipping that part.")

    with open("dpo_pairs.jsonl", "w") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")

    print(f"\nTotal pairs written: {len(pairs)}")
    print("Saved to dpo_pairs.jsonl")


if __name__ == "__main__":
    main()

Built 300 pairs from real labeled papers.
Added 133 pairs from synthetic negatives.

Total pairs written: 433
Saved to dpo_pairs.jsonl


In [10]:
import json
from collections import Counter
pairs = [json.loads(l) for l in open('dpo_pairs.jsonl')]
rejected_counts = Counter(p['rejected'] for p in pairs)
print('Total pairs:', len(pairs))
print('Unique rejected strings:', len(rejected_counts))
print('Most repeated:', rejected_counts.most_common(3))

Total pairs: 433
Unique rejected strings: 432
Most repeated: [('Irrelevant. source: perilla meal; modification step: physical: ultrasonication; application: solubility (downstream product property, not the isolated protein)', 2), ('Irrelevant. source: sesame; modification step: enzymatic: Alcalase; application: Water holding capacity (downstream product property, not the isolated protein)', 1), ('Irrelevant. source: soybean; modification step: physical: high hydrostatic pressure; application: Water holding capacity (downstream product property, not the isolated protein)', 1)]


In [9]:
import json
from collections import Counter

pairs = [json.loads(l) for l in open('dpo_pairs.jsonl')]
rejected_counts = Counter(p['rejected'] for p in pairs)

# distribution shape: how many strings appear only once vs many times
appearing_once = sum(1 for c in rejected_counts.values() if c == 1)
appearing_2_to_5 = sum(1 for c in rejected_counts.values() if 2 <= c <= 5)
appearing_6_plus = sum(1 for c in rejected_counts.values() if c >= 6)

print(f"Strings appearing exactly once: {appearing_once}")
print(f"Strings appearing 2-5 times: {appearing_2_to_5}")
print(f"Strings appearing 6+ times: {appearing_6_plus}")

Strings appearing exactly once: 431
Strings appearing 2-5 times: 1
Strings appearing 6+ times: 0
